In [23]:
import librosa
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib
import warnings

In [13]:
warnings.filterwarnings('ignore')

In [14]:
df = pd.read_csv(r"C:\Users\Arup Sarkar\Documents\EmoWave\data\dataemotion_data.csv")

In [15]:
df

,filename,emotion,intensity,statement,repetition,path
0,03-01-01-01-01-01-01.wav,neutral,normal,Kids are talking by the door,1st repetition,C:\Users\Arup Sarkar\Documents\EmoWave\data\Ac...
1,03-01-01-01-01-02-01.wav,neutral,normal,Kids are talking by the door,2nd repetition,C:\Users\Arup Sarkar\Documents\EmoWave\data\Ac...
2,03-01-01-01-02-01-01.wav,neutral,normal,Dogs are sitting by the door,1st repetition,C:\Users\Arup Sarkar\Documents\EmoWave\data\Ac...
3,03-01-01-01-02-02-01.wav,neutral,normal,Dogs are sitting by the door,2nd repetition,C:\Users\Arup Sarkar\Documents\EmoWave\data\Ac...
4,03-01-02-01-01-01-01.wav,calm,normal,Kids are talking by the door,1st repetition,C:\Users\Arup Sarkar\Documents\EmoWave\data\Ac...
...,...,...,...,...,...,...
1435,03-01-08-01-02-02-24.wav,surprised,normal,Dogs are sitting by the door,2nd repetition,C:\Users\Arup Sarkar\Documents\EmoWave\data\Ac...
1436,03-01-08-02-01-01-24.wav,surprised,strong,Kids are talking by the door,1st repetition,C:\Users\Arup Sarkar\Documents\EmoWave\data\Ac...
1437,03-01-08-02-01-02-24.wav,surprised,strong,Kids are talking by the door,2nd repetition,C:\Users\Arup Sarkar\Documents\EmoWave\data\Ac...
1438,03-01-08-02-02-01-24.wav,surprised,strong,Dogs are sitting by the door,1st repetition,C:\Users\Arup Sarkar\Documents\EmoWave\data\Ac...


In [18]:
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=None)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfccs, axis=1)
    mfcc_std = np.std(mfccs, axis=1)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)

    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    centroid_mean = np.mean(spectral_centroid)

    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_mean = np.mean(zcr)

    return np.hstack([mfcc_mean, mfcc_std, chroma_mean, centroid_mean, zcr_mean])

features = []
labels = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    file_path = row['path']
    feat = extract_features(file_path)
    features.append(feat)
    labels.append(row['emotion'])

X = np.array(features)
y = np.array(labels)
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

100%|██████████| 1440/1440 [01:07<00:00, 21.23it/s]

X shape: (1440, 40)
y shape: (1440,)


In [19]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'Train size: {X_train.shape}')
print(f'Test size: {X_test.shape}')
print(f'Classes: {len(le.classes_)}')

Train size: (1152, 40)
Test size: (288, 40)
Classes: 8


In [20]:
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42, probability=True)
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.6875
              precision    recall  f1-score   support

       angry       0.68      0.89      0.77        38
        calm       0.73      0.79      0.76        38
   disgusted       0.69      0.71      0.70        38
     fearful       0.60      0.72      0.65        39
       happy       0.79      0.56      0.66        39
     neutral       0.46      0.63      0.53        19
         sad       0.75      0.55      0.64        38
   surprised       0.83      0.62      0.71        39

    accuracy                           0.69       288
   macro avg       0.69      0.68      0.68       288
weighted avg       0.71      0.69      0.69       288



In [22]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

xgb = XGBClassifier(n_estimators=200, random_state=42, eval_metric='mlogloss')
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)

print(f'SVM accuracy:          {acc:.4f}')
print(f'Random Forest accuracy: {rf_acc:.4f}')
print(f'XGBoost accuracy:       {xgb_acc:.4f}')

SVM accuracy:          0.6875
Random Forest accuracy: 0.6250
XGBoost accuracy:       0.6042


In [24]:
os.makedirs('../models', exist_ok=True)

joblib.dump(svm, '../models/svm_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(le, '../models/label_encoder.pkl')

print('SVM model saved')
print('Scaler saved')
print('Label encoder saved')

SVM model saved
Scaler saved
Label encoder saved


## RE-EVALUATING MODELS WITH COMBINED FEATURES

In [25]:
combined_df = pd.read_csv(r"C:\Users\Arup Sarkar\Documents\EmoWave\data\combine_emotion_data.csv")

In [27]:
def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=None)

    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfccs, axis=1)
    mfcc_std = np.std(mfccs, axis=1)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    chroma_mean = np.mean(chroma, axis=1)

    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    centroid_mean = np.mean(spectral_centroid)

    zcr = librosa.feature.zero_crossing_rate(y)
    zcr_mean = np.mean(zcr)

    return np.hstack([mfcc_mean, mfcc_std, chroma_mean, centroid_mean, zcr_mean])

features = []
labels = []

for idx, row in tqdm(combined_df.iterrows(), total=len(combined_df)):
    file_path = row['path']
    feat = extract_features(file_path)
    features.append(feat)
    labels.append(row['emotion'])

X = np.array(features)
y = np.array(labels)
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

100%|██████████| 4048/4048 [01:36<00:00, 42.10it/s]

X shape: (4048, 40)
y shape: (4048,)


In [28]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f'Train size: {X_train.shape}')
print(f'Test size: {X_test.shape}')
print(f'Classes: {len(le.classes_)}')

Train size: (3238, 40)
Test size: (810, 40)
Classes: 7


In [29]:
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42, probability=True)
svm.fit(X_train, y_train)

y_pred = svm.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy: 0.8815
              precision    recall  f1-score   support

       angry       0.90      0.92      0.91       118
   disgusted       0.90      0.89      0.89       118
     fearful       0.92      0.80      0.86       119
       happy       0.85      0.87      0.86       119
     neutral       0.90      0.93      0.92        99
         sad       0.90      0.89      0.89       119
   surprised       0.81      0.87      0.84       118

    accuracy                           0.88       810
   macro avg       0.88      0.88      0.88       810
weighted avg       0.88      0.88      0.88       810



In [30]:
print(classification_report(y_test, y_pred, target_names=le.classes_))
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

xgb = XGBClassifier(n_estimators=200, random_state=42, eval_metric='mlogloss')
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_pred)

print(f'SVM accuracy:          {acc:.4f}')
print(f'Random Forest accuracy: {rf_acc:.4f}')
print(f'XGBoost accuracy:       {xgb_acc:.4f}')

              precision    recall  f1-score   support

       angry       0.90      0.92      0.91       118
   disgusted       0.90      0.89      0.89       118
     fearful       0.92      0.80      0.86       119
       happy       0.85      0.87      0.86       119
     neutral       0.90      0.93      0.92        99
         sad       0.90      0.89      0.89       119
   surprised       0.81      0.87      0.84       118

    accuracy                           0.88       810
   macro avg       0.88      0.88      0.88       810
weighted avg       0.88      0.88      0.88       810

SVM accuracy:          0.8815
Random Forest accuracy: 0.8815
XGBoost accuracy:       0.8679
